In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

def lakehouse_id(name):
    """Resolve a lakehouse in this workspace by name, so the notebook works in any workspace."""
    try:
        item = notebookutils.lakehouse.get(name)
        return item["id"] if isinstance(item, dict) else item.id
    except Exception:
        import sempy.fabric as fabric
        return fabric.resolve_item_id(name, type="Lakehouse")


SILVER_LAKEHOUSE_ID = lakehouse_id("Strava_Silver")
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
SILVER_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_LAKEHOUSE_ID}/Tables"

# Gold = star schema (PascalCase) for Power BI and the data agent
GOLD_DIM_DATE = "DimDate"
GOLD_DIM_ACTIVITY_TYPE = "DimActivityType"
GOLD_DIM_USER = "DimUser"
GOLD_DIM_SOURCE_SYSTEM = "DimSourceSystem"
GOLD_ACTIVITIES = "FactWorkout"
GOLD_DAILY = "FactTrainingDay"
GOLD_WEEKLY = "FactTrainingWeek"
GOLD_NUTRITION = "FactNutritionRecommendation"
GOLD_SAMPLES = "FactWorkoutSample"
GOLD_SEGMENTS = "FactWorkoutSegment"
GOLD_ZONES = "FactWorkoutZoneTime"
GOLD_BEST_EFFORTS = "FactWorkoutBestEffort"
GOLD_SECTIONS = "FactWorkoutSection"
GOLD_INSIGHTS = "FactWorkoutInsight"
GOLD_DIM_YEAR = "DimYear"
GOLD_FITNESS_YEAR = "FactFitnessYear"
GOLD_POWER_CURVE = "FactPowerCurveYear"
GOLD_FITNESS_DAILY = "FactFitnessDaily"
GOLD_FITNESS_SUMMARY = "FactFitnessSummary"
POWER_CURVE_DURATIONS = [(60, "1 min"), (300, "5 min"), (600, "10 min"), (900, "15 min"), (1200, "20 min"), (3600, "60 min")]
FTP_FACTOR = 0.95
FITNESS_YEARS_BACK = 2  # Fitness & FTP page: current year + 2 previous years
SAMPLE_BUCKET_SECONDS = 15
LEGACY_GOLD_TABLES = ["activities", "daily_summary", "training_weekly_metrics", "nutrition_recommendations"]


def read_silver(table_name):
    return spark.read.format("delta").load(f"{SILVER_TABLES}/{table_name}")


def save_gold(df, table_name, partition_col=None):
    writer = df.write.mode("overwrite").format("delta").option("overwriteSchema", "true")
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.save(f"Tables/{table_name}")

# Distance bands per sport (cycling uses longer bands than running/walking)
RIDE_BANDS = [(10, "<10 km"), (30, "10-30 km"), (50, "30-50 km"), (100, "50-100 km"), (None, ">100 km")]
FOOT_BANDS = [(5, "<5 km"), (10, "5-10 km"), (21.1, "10-21 km"), (42.2, "21-42 km"), (None, "42+ km")]


def _band(family, km, as_sort):
    def chain(bands):
        expr = None
        for i, (limit, label) in enumerate(bands, start=1):
            value = F.lit(i) if as_sort else F.lit(label)
            if limit is None:
                return expr.otherwise(value)
            expr = F.when(km < limit, value) if expr is None else expr.when(km < limit, value)
    return F.when(km.isNull() | (km <= 0), F.lit(0 if as_sort else "No distance")) \
            .when(family == "cycling", chain(RIDE_BANDS)).otherwise(chain(FOOT_BANDS))


def distance_category(family, km):
    return _band(family, km, False)


def distance_category_sort(family, km):
    return _band(family, km, True)

print("Starting Silver lakehouse -> Gold lakehouse aggregation")

In [ ]:
# Source-agnostic Silver workouts (primary record per duplicate group) mapped to the column set used by the Gold tables.
workouts = read_silver("Workout").where("is_primary_record")
# Intensity is the workout's average HR relative to the athlete's max HR - not to the workout's own peak, which would
# make every steady ride look hard.
athlete_max_hr = (workouts.where(F.col("max_heart_rate_bpm").between(120, 230))
                  .agg(F.max("max_heart_rate_bpm")).collect()[0][0]) or 190.0

silver = (
    workouts
    .withColumn("activity_type_code", F.col("activity_type"))
    .withColumn("activity_key", F.concat_ws(":", "source_system", "source_workout_id"))
    .withColumn("source_activity_id", F.col("source_workout_id"))
    .withColumn("activity_name", F.col("workout_name"))
    .withColumn("activity_type", F.coalesce(F.get_json_object("source_attributes_json", "$.strava_type"), F.col("source_activity_type")))
    .withColumn("sport_type", F.col("source_activity_type"))
    .withColumn("activity_start_local", F.col("start_time_local"))
    .withColumn("activity_date", F.col("local_date"))
    .withColumn("distance_meters", F.col("distance_m"))
    .withColumn("moving_seconds", F.col("moving_duration_s"))
    .withColumn("elapsed_seconds", F.col("elapsed_duration_s"))
    .withColumn("avg_power_watts", F.col("avg_power_w"))
    .withColumn("max_power_watts", F.col("max_power_w"))
    .withColumn("normalized_power_watts", F.col("normalized_power_w"))
    .withColumn("elevation_gain_meters", F.col("elevation_gain_m"))
    .withColumn("elevation_high_meters", F.col("elevation_high_m"))
    .withColumn("elevation_low_meters", F.col("elevation_low_m"))
    .withColumn("kilojoules", F.col("energy_kj"))
    .withColumn("training_load", F.col("training_load_score"))
    .withColumn("cadence_rpm", F.col("avg_cadence"))
    .withColumn("distance_km", F.round(F.col("distance_m") / F.lit(1000.0), 3))
    .withColumn("duration_minutes", F.round(F.col("moving_duration_s") / F.lit(60.0), 2))
    .withColumn("duration_hours", F.round(F.col("moving_duration_s") / F.lit(3600.0), 3))
    .withColumn("avg_speed_kmh", F.round(F.col("avg_speed_mps") * F.lit(3.6), 2))
    .withColumn("intensity_ratio", F.coalesce(F.col("hr_intensity_ratio"), F.col("avg_heart_rate_bpm") / F.lit(float(athlete_max_hr))))
    .withColumn(
        "intensity_zone",
        F.when(F.col("intensity_ratio").isNull(), F.lit("No heart rate"))
         .when(F.col("intensity_ratio") >= 0.9, F.lit("Zone 5 - Max"))
         .when(F.col("intensity_ratio") >= 0.8, F.lit("Zone 4 - Hard"))
         .when(F.col("intensity_ratio") >= 0.7, F.lit("Zone 3 - Tempo"))
         .when(F.col("intensity_ratio") >= 0.6, F.lit("Zone 2 - Endurance"))
         .otherwise(F.lit("Zone 1 - Easy"))
    )
    .withColumn(
        "optimal_carbs_g",
        F.when(F.col("duration_hours") < 1, F.lit(0.0))
         .when((F.col("duration_hours") < 1.5) & (F.col("intensity_ratio") < 0.75), F.col("duration_hours") * 30)
         .when((F.col("duration_hours") < 1.5) & (F.col("intensity_ratio") >= 0.75), F.col("duration_hours") * 45)
         .when((F.col("duration_hours") < 2.5) & (F.col("intensity_ratio") < 0.75), F.col("duration_hours") * 45)
         .when((F.col("duration_hours") < 2.5) & (F.col("intensity_ratio") >= 0.75), F.col("duration_hours") * 60)
         .when((F.col("duration_hours") >= 2.5) & (F.col("intensity_ratio") < 0.75), F.col("duration_hours") * 60)
         .when((F.col("duration_hours") >= 2.5) & (F.col("intensity_ratio") >= 0.75), F.col("duration_hours") * 90)
         .otherwise(F.lit(0.0))
    )
)

gold_activities = (
    silver
    .select(
        F.col("source_activity_id").cast("long").alias("id"),
        F.col("activity_name").alias("name"),
        F.col("activity_type").alias("type"),
        "sport_type",
        F.date_format("activity_start_local", "yyyy-MM-dd'T'HH:mm:ss").alias("start_date_local"),
        F.col("distance_meters").alias("distance"),
        F.col("moving_seconds").alias("moving_time"),
        F.col("elapsed_seconds").alias("elapsed_time"),
        F.col("avg_speed_mps").alias("average_speed"),
        F.col("max_speed_mps").alias("max_speed"),
        F.col("avg_power_watts").alias("average_watts"),
        F.col("max_power_watts").alias("max_watts"),
        F.col("normalized_power_watts").alias("weighted_average_watts"),
        F.col("avg_heart_rate_bpm").alias("average_heartrate"),
        F.col("max_heart_rate_bpm").alias("max_heartrate"),
        F.col("avg_temperature_c").alias("average_temp"),
        F.col("elevation_gain_meters").alias("total_elevation_gain"),
        F.col("elevation_high_meters").alias("elev_high"),
        F.col("elevation_low_meters").alias("elev_low"),
        "kilojoules",
        F.col("source_effort_score").alias("suffer_score"),
        F.col("cadence_rpm").alias("average_cadence"),
        F.col("is_trainer").alias("trainer"),
        F.col("is_commute").alias("commute"),
        "distance_km",
        "avg_speed_kmh",
        "duration_minutes",
        "activity_date",
        "intensity_zone",
        "optimal_carbs_g",
        "source_system",
        "activity_key",
        F.col("workout_id").alias("workout_key"),
        F.col("user_id").alias("user_key"),
        F.col("activity_type_code").alias("activity_type_key"),
        F.col("activity_date").alias("date_key"),
        "training_load_score",
        "training_load_method",
        "tss",
        "intensity_factor",
        "is_indoor",
        F.when(F.col("is_indoor"), F.lit("Indoor")).otherwise(F.lit("Outdoor")).alias("environment"),
        distance_category(F.col("activity_family"), F.col("distance_km")).alias("distance_category"),
        distance_category_sort(F.col("activity_family"), F.col("distance_km")).alias("distance_category_sort"),
        F.col("source_system").alias("source_system_key"),
        F.col("activity_family"),
        F.initcap(F.regexp_replace("activity_family", "_", " ")).alias("activity_family_name"),
        F.initcap(F.regexp_replace("activity_type_code", "_", " ")).alias("activity_type_name"),
        F.concat_ws(" · ", F.date_format("activity_start_local", "yyyy-MM-dd"), F.coalesce("activity_name", F.lit("Workout")),
                    F.initcap(F.regexp_replace("activity_family", "_", " "))).alias("activity_label"),
        F.col("activity_start_local").alias("start_time_local"),
        F.hour("activity_start_local").alias("start_hour"),
        "duration_hours",
        F.round("active_energy_kcal", 0).alias("calories_kcal"),
        "perceived_exertion",
        "device_name",
        "workout_intent",
        F.when(F.col("distance_m") > 0, F.round(F.col("moving_duration_s") / 60.0 / (F.col("distance_m") / 1000.0), 2)).alias("pace_min_per_km"),
    )
)

daily_summary = (
    silver
    .groupBy("activity_date")
    .agg(
        F.count("*").cast("int").alias("workouts"),
        F.concat_ws(",", F.array_sort(F.collect_set("source_system"))).alias("source_systems"),
        F.countDistinct("source_system").cast("int").alias("source_count"),
        F.round(F.sum("distance_km"), 2).alias("distance_km"),
        F.round(F.sum("duration_hours"), 2).alias("training_hours"),
        F.round(F.sum("elevation_gain_meters"), 0).alias("elevation_gain_m"),
        F.round(F.avg("avg_heart_rate_bpm"), 1).alias("avg_heart_rate"),
        F.round(F.sum(F.coalesce("training_load", F.lit(0.0))), 1).alias("training_load"),
        F.round(F.sum(F.coalesce("kilojoules", F.lit(0.0))), 1).alias("kilojoules"),
        F.round(F.sum(F.coalesce("optimal_carbs_g", F.lit(0.0))), 1).alias("optimal_carbs_g"),
    )
    .orderBy("activity_date")
)

weekly = (
    silver
    .withColumn("week_start", F.date_trunc("week", F.col("activity_date")).cast("date"))
    .groupBy("week_start")
    .agg(
        F.count("*").cast("int").alias("workouts"),
        F.concat_ws(",", F.array_sort(F.collect_set("source_system"))).alias("source_systems"),
        F.countDistinct("source_system").cast("int").alias("source_count"),
        F.round(F.sum("distance_km"), 2).alias("distance_km"),
        F.round(F.sum("duration_hours"), 2).alias("training_hours"),
        F.round(F.sum("elevation_gain_meters"), 0).alias("elevation_gain_m"),
        F.round(F.avg("avg_heart_rate_bpm"), 1).alias("avg_heart_rate"),
        F.round(F.avg("avg_power_watts"), 1).alias("avg_watts"),
        F.round(F.sum(F.coalesce("training_load", F.lit(0.0))), 1).alias("training_load"),
        F.round(F.sum(F.coalesce("kilojoules", F.lit(0.0))), 1).alias("kilojoules"),
        F.round(F.sum(F.coalesce("optimal_carbs_g", F.lit(0.0))), 1).alias("optimal_carbs_g"),
        F.sum(F.when(F.col("activity_type").isin("Ride", "VirtualRide"), 1).otherwise(0)).cast("int").alias("rides"),
        F.sum(F.when(F.col("activity_type") == "Run", 1).otherwise(0)).cast("int").alias("runs"),
        F.sum(F.when(F.col("activity_type") == "Swim", 1).otherwise(0)).cast("int").alias("swims"),
        F.sum(F.when(F.col("activity_type").isin("WeightTraining", "Workout"), 1).otherwise(0)).cast("int").alias("strength_workouts"),
        F.sum(F.when(F.col("intensity_zone") == "Zone 1 - Easy", F.col("duration_hours")).otherwise(0.0)).alias("zone1_hours"),
        F.sum(F.when(F.col("intensity_zone") == "Zone 2 - Endurance", F.col("duration_hours")).otherwise(0.0)).alias("zone2_hours"),
        F.sum(F.when(F.col("intensity_zone") == "Zone 3 - Tempo", F.col("duration_hours")).otherwise(0.0)).alias("zone3_hours"),
        F.sum(F.when(F.col("intensity_zone") == "Zone 4 - Hard", F.col("duration_hours")).otherwise(0.0)).alias("zone4_hours"),
        F.sum(F.when(F.col("intensity_zone") == "Zone 5 - Max", F.col("duration_hours")).otherwise(0.0)).alias("zone5_hours"),
    )
)

rolling_window = Window.orderBy("week_start").rowsBetween(-3, 0)
previous_window = Window.orderBy("week_start")
weekly_metrics = (
    weekly
    .withColumn("load_4w_avg", F.round(F.avg("training_load").over(rolling_window), 1))
    .withColumn("hours_4w_avg", F.round(F.avg("training_hours").over(rolling_window), 2))
    .withColumn("distance_4w_avg", F.round(F.avg("distance_km").over(rolling_window), 2))
    .withColumn("previous_week_load", F.lag("training_load").over(previous_window))
    .withColumn("load_change_pct", F.round((F.col("training_load") - F.col("previous_week_load")) / F.col("previous_week_load") * 100.0, 1))
    .withColumn(
        "readiness_flag",
        F.when(F.col("load_change_pct") >= 25, F.lit("Ramp too fast"))
         .when(F.col("load_change_pct") <= -30, F.lit("Recovery / down week"))
         .otherwise(F.lit("Steady"))
    )
    .withColumn("easy_hours", F.round(F.col("zone1_hours") + F.col("zone2_hours"), 2))
    .withColumn("tempo_hours", F.round(F.col("zone3_hours"), 2))
    .withColumn("hard_hours", F.round(F.col("zone4_hours") + F.col("zone5_hours"), 2))
    .select(
        "week_start", "workouts", "source_systems", "source_count", "distance_km", "training_hours", "elevation_gain_m",
        "avg_heart_rate", "avg_watts", "training_load", "load_4w_avg", "load_change_pct",
        "readiness_flag", "kilojoules", "optimal_carbs_g", "rides", "runs", "swims",
        "strength_workouts", "easy_hours", "tempo_hours", "hard_hours", "hours_4w_avg", "distance_4w_avg"
    )
    .orderBy("week_start")
)

nutrition_recommendations = (
    daily_summary
    .withColumn(
        "fueling_focus",
        F.when(F.col("training_hours") >= 2.0, F.lit("Long session fueling"))
         .when(F.col("training_hours") >= 1.0, F.lit("Moderate workout fueling"))
         .otherwise(F.lit("Baseline nutrition"))
    )
    .withColumn(
        "recommendation",
        F.when(F.col("training_hours") >= 2.0, F.concat(F.lit("Plan "), F.col("optimal_carbs_g").cast("int"), F.lit("g carbs during/around training and prioritize recovery protein.")))
         .when(F.col("training_hours") >= 1.0, F.concat(F.lit("Use "), F.col("optimal_carbs_g").cast("int"), F.lit("g carbs around the workout if intensity is moderate or higher.")))
         .otherwise(F.lit("Focus on regular meals, hydration, and protein distribution."))
    )
)

# Conformed dimensions
date_bounds = silver.agg(F.min("activity_date").alias("d0")).collect()[0]
dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{date_bounds['d0']}'), date_add(current_date(), 60))) AS date_key")
    .select(
        "date_key",
        F.year("date_key").alias("year"),
        F.quarter("date_key").alias("quarter"),
        F.month("date_key").alias("month"),
        F.date_format("date_key", "MMM").alias("month_name"),
        F.concat(F.date_format("date_key", "MMM"), F.lit(" '"), F.date_format("date_key", "yy")).alias("month_year_label"),
        (F.year("date_key") * 100 + F.month("date_key")).alias("year_month_sort"),
        F.date_trunc("week", "date_key").cast("date").alias("week_start"),
        F.weekofyear("date_key").alias("iso_week"),
        F.dayofweek("date_key").alias("day_of_week"),
        ((F.dayofweek("date_key") + 5) % 7 + 1).alias("weekday_number"),
        F.date_format("date_key", "EEE").alias("day_name"),
        F.dayofweek("date_key").isin(1, 7).alias("is_weekend"),
    )
)
type_rank = Window.partitionBy("activity_type").orderBy(F.when(F.col("source_system") == "strava", 0).otherwise(1), "source_activity_type")
dim_activity_type = (
    read_silver("ActivityTypeMapping")
    .withColumn("_rn", F.row_number().over(type_rank)).where("_rn = 1")
    .select(F.col("activity_type").alias("activity_type_key"), "activity_family", "is_endurance", "is_distance_based",
            "is_indoor_default", "primary_intensity_metric",
            F.initcap(F.regexp_replace("activity_type", "_", " ")).alias("activity_type_name"),
            F.initcap(F.regexp_replace("activity_family", "_", " ")).alias("activity_family_name"))
    .unionByName(spark.createDataFrame(
        [("other", "other", False, False, False, "heart_rate", "Other", "Other")],
        "activity_type_key string, activity_family string, is_endurance boolean, is_distance_based boolean, "
        "is_indoor_default boolean, primary_intensity_metric string, activity_type_name string, activity_family_name string"))
    .dropDuplicates(["activity_type_key"])
)
dim_user = read_silver("UserProfile").select(
    F.col("user_id").alias("user_key"), "display_name", "sex", "home_country", "measurement_system", "primary_timezone_name")

save_gold(dim_date, GOLD_DIM_DATE)
save_gold(dim_activity_type, GOLD_DIM_ACTIVITY_TYPE)
dim_source_system = read_silver("DataSource").select(
    F.col("source_system").alias("source_system_key"),
    "source_name", "source_category", "integration_method", "data_domains_json",
    "workout_priority", "is_active", "bronze_schema", "load_order",
)

save_gold(dim_user, GOLD_DIM_USER)
save_gold(dim_source_system, GOLD_DIM_SOURCE_SYSTEM)
save_gold(gold_activities, GOLD_ACTIVITIES, "activity_date")
save_gold(daily_summary.withColumn("date_key", F.col("activity_date")), GOLD_DAILY)
save_gold(weekly_metrics, GOLD_WEEKLY)
save_gold(nutrition_recommendations.withColumn("date_key", F.col("activity_date")), GOLD_NUTRITION)

# Activity-level detail for the single-activity analysis page (primary workout records only)
primary_keys = workouts.select("workout_id")


def pace_label(minutes_col):
    total_s = F.round(minutes_col * 60).cast("int")
    return F.when(minutes_col.isNotNull(), F.format_string("%d:%02d", F.floor(total_s / 60).cast("int"), (total_s % 60).cast("int")))


def duration_label(seconds_col):
    s_ = seconds_col.cast("int")
    return F.when(s_ >= 3600, F.format_string("%d:%02d:%02d", (s_ / 3600).cast("int"), ((s_ % 3600) / 60).cast("int"), (s_ % 60).cast("int"))) \
            .otherwise(F.format_string("%d:%02d", (s_ / 60).cast("int"), (s_ % 60).cast("int")))


samples = (
    read_silver("WorkoutSample").join(primary_keys, "workout_id")
    .withColumn("bucket_s", (F.floor(F.col("sample_offset_s") / SAMPLE_BUCKET_SECONDS) * SAMPLE_BUCKET_SECONDS).cast("int"))
    .groupBy("workout_id", "bucket_s")
    .agg(
        F.round(F.max("distance_m") / 1000.0, 3).alias("distance_km"),
        F.round(F.avg("heart_rate_bpm"), 1).alias("heart_rate_bpm"),
        F.round(F.avg("power_w"), 1).alias("power_w"),
        F.round(F.avg("speed_mps") * 3.6, 2).alias("speed_kmh"),
        F.round(F.avg("cadence"), 1).alias("cadence"),
        F.round(F.avg("altitude_m"), 1).alias("altitude_m"),
        F.round(F.avg("grade_pct"), 1).alias("grade_pct"),
    )
    .select(F.col("workout_id").alias("workout_key"), F.col("bucket_s").alias("elapsed_s"),
            F.round(F.col("bucket_s") / 60.0, 2).alias("elapsed_min"), "distance_km", "heart_rate_bpm", "power_w",
            "speed_kmh", "cadence", "altitude_m", "grade_pct")
)

segments = (
    read_silver("WorkoutSegment").join(primary_keys, "workout_id")
    .withColumn("km", F.col("distance_m") / 1000.0)
    .withColumn("pace", F.when(F.col("km") > 0.05, F.col("moving_duration_s") / 60.0 / F.col("km")))
    .select(
        F.col("workout_id").alias("workout_key"),
        F.when(F.col("segment_type") == "lap", "Lap").otherwise("Km split").alias("segment_type"),
        F.col("segment_index").cast("int").alias("segment_index"),
        F.coalesce("segment_name", F.concat(F.lit("#"), F.col("segment_index"))).alias("segment_name"),
        F.round("km", 2).alias("distance_km"),
        F.round(F.col("moving_duration_s") / 60.0, 2).alias("moving_min"),
        duration_label(F.col("moving_duration_s")).alias("moving_time_label"),
        F.round("pace", 2).alias("pace_min_per_km"),
        pace_label(F.col("pace")).alias("pace_label"),
        F.round(F.col("avg_speed_mps") * 3.6, 1).alias("speed_kmh"),
        F.round(F.col("grade_adjusted_speed_mps") * 3.6, 1).alias("grade_adjusted_speed_kmh"),
        F.round("avg_heart_rate_bpm", 0).alias("avg_heart_rate_bpm"),
        F.round("max_heart_rate_bpm", 0).alias("max_heart_rate_bpm"),
        F.round("avg_power_w", 0).alias("avg_power_w"),
        F.round("avg_cadence", 0).alias("avg_cadence"),
        F.round(F.coalesce("elevation_gain_m", "elevation_change_m"), 0).alias("elevation_m"),
    )
)

zones = (
    read_silver("WorkoutZoneTime").join(primary_keys, "workout_id")
    .select(
        F.col("workout_id").alias("workout_key"),
        F.when(F.col("zone_type") == "heart_rate", "Heart rate").otherwise("Power").alias("zone_type"),
        F.col("zone_number").cast("int").alias("zone_number"),
        F.concat(F.lit("Z"), F.col("zone_number")).alias("zone_label"),
        F.concat(F.lit("Z"), F.col("zone_number"), F.lit(" "), F.col("zone_name")).alias("zone_display"),
        "zone_name",
        F.round(F.col("seconds_in_zone") / 60.0, 1).alias("minutes"),
        "pct_of_zone_time",
    )
)

efforts_raw = (
    read_silver("WorkoutBestEffort").join(primary_keys, "workout_id")
    .join(workouts.select("workout_id", "activity_family"), "workout_id")
)
best_window = Window.partitionBy("effort_type", "effort_name", "activity_family")
best_efforts = (
    efforts_raw
    .withColumn("best_power_w", F.max("avg_power_w").over(best_window))
    .withColumn("best_duration_s", F.min("duration_s").over(best_window))
    .withColumn("is_power", F.col("effort_type") == "power_duration")
    .select(
        F.col("workout_id").alias("workout_key"),
        F.when(F.col("is_power"), "Power").otherwise("Distance").alias("effort_type"),
        F.regexp_replace("effort_name", " power$", "").alias("effort_name"),
        F.coalesce(F.when(F.col("is_power"), F.col("duration_s")), F.round(F.col("distance_m"))).cast("long").alias("effort_sort"),
        "duration_s",
        duration_label(F.col("duration_s")).alias("time_label"),
        F.round("avg_power_w", 0).alias("avg_power_w"),
        F.round("best_power_w", 0).alias("best_power_w"),
        F.when(F.col("is_power"), F.concat(F.round("avg_power_w", 0).cast("int").cast("string"), F.lit(" W")))
         .otherwise(duration_label(F.col("duration_s"))).alias("result_label"),
        F.when(F.col("is_power"), F.concat(F.round("best_power_w", 0).cast("int").cast("string"), F.lit(" W")))
         .otherwise(duration_label(F.col("best_duration_s"))).alias("best_label"),
        F.round(F.when(F.col("is_power"), F.col("avg_power_w") / F.col("best_power_w"))
                 .otherwise(F.col("best_duration_s") / F.col("duration_s")) * 100, 1).alias("pct_of_best"),
        (F.when(F.col("is_power"), F.col("avg_power_w") >= F.col("best_power_w"))
          .otherwise(F.col("duration_s") <= F.col("best_duration_s"))).alias("is_all_time_best"),
    )
)

save_gold(samples, GOLD_SAMPLES)
save_gold(segments, GOLD_SEGMENTS)
save_gold(zones, GOLD_ZONES)
save_gold(best_efforts, GOLD_BEST_EFFORTS)

# ---------------------------------------------------------------- Sections (10 km, or 1 km for activities < 20 km)
families = workouts.select("workout_id", "activity_family")
ts = (
    read_silver("WorkoutSample").join(primary_keys, "workout_id").join(families, "workout_id")
    .where(F.col("distance_m").isNotNull())
)
w_all = Window.partitionBy("workout_id")
w_ord = Window.partitionBy("workout_id").orderBy("sample_offset_s")
ts = (
    ts.withColumn("total_m", F.max("distance_m").over(w_all))
    .where(F.col("total_m") >= 500)
    .withColumn("block_m", F.when(F.col("total_m") >= 20000, F.lit(10000)).otherwise(F.lit(1000)))
    .withColumn("moving", F.coalesce("is_moving", F.lit(True)))
    .withColumn("dist_diff", F.greatest(F.col("distance_m") - F.lag("distance_m").over(w_ord), F.lit(0.0)))
    .withColumn("alt_diff", F.col("altitude_m") - F.lag("altitude_m").over(w_ord))
    .withColumn("block_index", F.least(F.floor(F.col("distance_m") / F.col("block_m")),
                                       F.floor((F.col("total_m") - 1) / F.col("block_m"))).cast("int"))
    .withColumn("half", F.when(F.col("distance_m") <= F.col("total_m") / 2, F.lit(1)).otherwise(F.lit(2)))
)
km_label = lambda c: F.regexp_replace(F.format_string("%.1f", c), "\\.0$", "")
section_base = (
    ts.groupBy("workout_id", "activity_family", "block_m", "block_index", "total_m")
    .agg(
        F.sum("dist_diff").alias("dist_m"),
        F.sum(F.when(F.col("moving"), F.col("sample_duration_s"))).alias("moving_s"),
        F.avg(F.when(F.col("moving"), F.col("heart_rate_bpm"))).alias("hr"),
        F.avg(F.when(F.col("moving"), F.col("power_w"))).alias("pw"),
        F.sum(F.when(F.col("alt_diff") > 0, F.col("alt_diff"))).alias("gain"),
    )
    .withColumn("start_km", F.col("block_index") * F.col("block_m") / 1000.0)
    .withColumn("end_km", F.least((F.col("block_index") + 1) * F.col("block_m"), F.col("total_m")) / 1000.0)
    .withColumn("speed_kmh", F.when(F.col("moving_s") > 0, F.col("dist_m") / F.col("moving_s") * 3.6))
    .withColumn("pace_min", F.when(F.col("speed_kmh") > 0, 60.0 / F.col("speed_kmh")))
    .withColumn("coverage", (F.col("end_km") - F.col("start_km")) * 1000.0 / F.col("block_m"))
)
sections = section_base.select(
    F.col("workout_id").alias("workout_key"),
    F.col("block_index").alias("section_index"),
    F.concat(km_label(F.col("start_km")), F.lit("–"), km_label(F.col("end_km")), F.lit(" km")).alias("section_label"),
    (F.col("block_m") / 1000).cast("int").alias("section_size_km"),
    F.round(F.col("dist_m") / 1000.0, 2).alias("distance_km"),
    duration_label(F.coalesce(F.col("moving_s"), F.lit(0))).alias("moving_time_label"),
    F.round("speed_kmh", 1).alias("avg_speed_kmh"),
    F.when(F.col("activity_family").isin("running", "walking"), pace_label(F.col("pace_min"))).alias("pace_label"),
    F.round("hr", 0).alias("avg_heart_rate_bpm"),
    F.round("pw", 0).alias("avg_power_w"),
    F.round(F.coalesce("gain", F.lit(0.0)), 0).alias("elevation_gain_m"),
)

# ---------------------------------------------------------------- Plain-language insights (one row per workout)
halves = (
    ts.groupBy("workout_id", "half")
    .agg(F.sum("dist_diff").alias("d"), F.sum(F.when(F.col("moving"), F.col("sample_duration_s"))).alias("t"),
         F.avg(F.when(F.col("moving"), F.col("heart_rate_bpm"))).alias("hr"))
    .withColumn("spd", F.when(F.col("t") > 0, F.col("d") / F.col("t") * 3.6))
    .groupBy("workout_id")
    .pivot("half", [1, 2])
    .agg(F.first("spd").alias("spd"), F.first("hr").alias("hr"))
)
pct = lambda c: F.abs(F.round(c, 0)).cast("int").cast("string")
spd_chg = (F.col("2_spd") - F.col("1_spd")) / F.col("1_spd") * 100
hr_chg = F.col("2_hr") - F.col("1_hr")
is_pace_sport = F.col("activity_family").isin("running", "walking")

fast_rank = Window.partitionBy("workout_id").orderBy(F.desc("speed_kmh"))
climb_rank = Window.partitionBy("workout_id").orderBy(F.desc("gain"))
fastest = (
    section_base.where(F.col("coverage") >= 0.5).withColumn("_r", F.row_number().over(fast_rank)).where("_r = 1")
    .select("workout_id",
            F.concat(F.lit("Fastest "), km_label(F.col("start_km")), F.lit("–"), km_label(F.col("end_km")), F.lit(" km: "),
                     F.when(F.col("activity_family").isin("running", "walking"), F.concat(pace_label(F.col("pace_min")), F.lit(" /km")))
                      .otherwise(F.concat(F.format_string("%.1f", F.col("speed_kmh")), F.lit(" km/h")))).alias("fastest_text"))
)
totals = section_base.groupBy("workout_id").agg(F.sum("gain").alias("total_gain"))
climb = (
    section_base.withColumn("_r", F.row_number().over(climb_rank)).where("_r = 1").join(totals, "workout_id")
    .select("workout_id",
            F.when(F.col("total_gain") < 30, F.concat(F.lit("Flat route (+"), F.round("total_gain", 0).cast("int").cast("string"), F.lit(" m in total)")))
             .otherwise(F.concat(F.lit("Most climbing "), km_label(F.col("start_km")), F.lit("–"), km_label(F.col("end_km")),
                                 F.lit(" km (+"), F.round("gain", 0).cast("int").cast("string"), F.lit(" m)"))).alias("climb_text"))
)
zone_rank = Window.partitionBy("workout_id").orderBy(F.desc("seconds_in_zone"))
intensity = (
    read_silver("WorkoutZoneTime").join(primary_keys, "workout_id").where(F.col("zone_type") == "heart_rate")
    .withColumn("_r", F.row_number().over(zone_rank)).where("_r = 1")
    .select("workout_id",
            F.concat(F.lit("Mostly Z"), F.col("zone_number"), F.lit(" "), F.col("zone_name"), F.lit(" ("),
                     F.round("pct_of_zone_time", 0).cast("int").cast("string"), F.lit("% of the time)")).alias("intensity_text"))
)
effort_rank = Window.partitionBy("workout_key").orderBy(
    F.when((F.col("effort_type") == "Power") & (F.col("effort_name") == "20 min"), 0).when(F.col("effort_type") == "Distance", 1).otherwise(2),
    F.desc("effort_sort"))
best_effort = (
    best_efforts.withColumn("_r", F.row_number().over(effort_rank)).where("_r = 1")
    .select(F.col("workout_key").alias("workout_id"),
            F.concat(F.lit("Best "), F.col("effort_name"), F.lit(": "), F.col("result_label"), F.lit(" ("),
                     F.round("pct_of_best", 0).cast("int").cast("string"), F.lit("% of your best)")).alias("best_effort_text"))
)
insights = (
    workouts.select("workout_id", "activity_family")
    .join(halves, "workout_id", "left").join(intensity, "workout_id", "left").join(fastest, "workout_id", "left")
    .join(climb, "workout_id", "left").join(best_effort, "workout_id", "left")
    .select(
        F.col("workout_id").alias("workout_key"),
        F.coalesce("intensity_text", F.lit("No heart-rate data")).alias("intensity_text"),
        F.when(F.col("1_spd").isNull() | F.col("2_spd").isNull() | (F.col("1_spd") <= 0), F.lit("No distance data"))
         .when(F.abs(spd_chg) < 3, F.concat(F.lit("Even pacing (2nd half "), F.when(spd_chg >= 0, "+").otherwise("-"), pct(spd_chg), F.lit("%)")))
         .when(spd_chg > 0, F.concat(F.lit("Finished stronger: 2nd half "), pct(spd_chg), F.lit("% faster")))
         .otherwise(F.concat(F.lit("Slowed down: 2nd half "), pct(spd_chg), F.lit("% slower"))).alias("pacing_text"),
        F.when(F.col("1_hr").isNull() | F.col("2_hr").isNull(), F.lit("No heart-rate data"))
         .when(F.abs(hr_chg) <= 3, F.concat(F.lit("Stable heart rate (~"), F.round((F.col("1_hr") + F.col("2_hr")) / 2, 0).cast("int").cast("string"), F.lit(" bpm)")))
         .when(hr_chg > 3, F.concat(F.lit("Heart rate rose +"), F.round(hr_chg, 0).cast("int").cast("string"), F.lit(" bpm in 2nd half (fatigue)")))
         .otherwise(F.concat(F.lit("Heart rate dropped "), F.round(-hr_chg, 0).cast("int").cast("string"), F.lit(" bpm in 2nd half"))).alias("heart_rate_text"),
        F.coalesce("fastest_text", F.lit("No distance data")).alias("fastest_text"),
        F.coalesce("climb_text", F.lit("No elevation data")).alias("climb_text"),
        F.coalesce("best_effort_text", F.lit("No power or PR data")).alias("best_effort_text"),
    )
)
save_gold(sections, GOLD_SECTIONS)
save_gold(insights, GOLD_INSIGHTS)

# ---------------------------------------------------------------- Fitness & FTP (cycling power curve, yearly fitness, CTL/ATL/TSB)
import datetime as _dt
today = _dt.date.today()
this_year, last_year = today.year, today.year - 1
wk = workouts.select("workout_id", "activity_family", "local_date", "workout_name", "max_heart_rate_bpm", "distance_m",
                     "moving_duration_s", "training_load_score").withColumn("year", F.year("local_date"))

all_efforts = read_silver("WorkoutBestEffort").join(primary_keys, "workout_id").join(wk, "workout_id")
ride_power = (
    all_efforts.where((F.col("effort_type") == "power_duration") & (F.col("activity_family") == "cycling")
                      & (F.col("avg_power_w") > 0) & (F.col("avg_power_w") < 2000))
)
dur_df = spark.createDataFrame(POWER_CURVE_DURATIONS, "duration_s int, duration_label string")
yr_rank = Window.partitionBy("year", "duration_s").orderBy(F.desc("avg_power_w"), F.desc("local_date"))
all_rank = Window.partitionBy("duration_s")
power_curve = (
    ride_power.join(F.broadcast(dur_df), "duration_s")
    .withColumn("_r", F.row_number().over(yr_rank)).where("_r = 1")
    .withColumn("all_time_best_w", F.max("avg_power_w").over(all_rank))
    .select(
        "year",
        F.col("year").cast("string").alias("year_label"),
        "duration_s", "duration_label",
        F.round("avg_power_w", 0).alias("best_power_w"),
        F.col("local_date").alias("effort_date"),
        F.coalesce("workout_name", F.lit("Ride")).alias("workout_name"),
        (F.col("avg_power_w") >= F.col("all_time_best_w")).alias("is_all_time_best"),
    )
)

def best_power(df, seconds, alias):
    return df.where(F.col("duration_s") == seconds).groupBy("year").agg(F.max("avg_power_w").alias(alias))

run_best = (
    all_efforts.where((F.col("effort_type") == "distance") & (F.col("activity_family") == "running") & (F.col("duration_s") > 0))
    .withColumn("dist_key", F.lower(F.regexp_replace("effort_name", "[^A-Za-z0-9]", "")))
    .groupBy("year").pivot("dist_key", ["5k", "10k"]).agg(F.min("duration_s"))
    .withColumnRenamed("5k", "best_5k_s").withColumnRenamed("10k", "best_10k_s")
)
yearly_activity = (
    wk.groupBy("year").agg(
        F.sum(F.when(F.col("activity_family") == "cycling", 1).otherwise(0)).cast("int").alias("rides"),
        F.round(F.sum(F.when(F.col("activity_family") == "cycling", F.col("moving_duration_s"))) / 3600.0, 1).alias("ride_hours"),
        F.round(F.sum(F.when(F.col("activity_family") == "cycling", F.col("distance_m"))) / 1000.0, 0).alias("ride_km"),
        F.sum(F.when(F.col("activity_family") == "running", 1).otherwise(0)).cast("int").alias("runs"),
        F.round(F.sum(F.when(F.col("activity_family") == "running", F.col("distance_m"))) / 1000.0, 0).alias("run_km"),
        F.count("*").cast("int").alias("workouts"),
        F.round(F.sum("moving_duration_s") / 3600.0, 0).alias("training_hours"),
        F.round(F.sum("training_load_score"), 0).alias("training_load"),
        F.max(F.when(F.col("max_heart_rate_bpm").between(100, 230), F.col("max_heart_rate_bpm"))).alias("max_hr_bpm"),
    )
)
fitness_year = (
    yearly_activity
    .join(best_power(ride_power, 1200, "best_20min_w"), "year", "left")
    .join(best_power(ride_power, 3600, "best_60min_w"), "year", "left")
    .join(best_power(ride_power, 300, "best_5min_w"), "year", "left")
    .join(run_best, "year", "left")
    .withColumn("ftp_w", F.round(F.col("best_20min_w") * FTP_FACTOR, 0))
    .withColumn("prev_ftp_w", F.lag("ftp_w").over(Window.orderBy("year")))
    .select(
        "year", F.col("year").cast("string").alias("year_label"),
        "ftp_w",
        F.round(F.when(F.col("prev_ftp_w") > 0, (F.col("ftp_w") - F.col("prev_ftp_w")) / F.col("prev_ftp_w")), 3).alias("ftp_change_pct"),
        F.round("best_5min_w", 0).alias("best_5min_w"), F.round("best_20min_w", 0).alias("best_20min_w"),
        F.round("best_60min_w", 0).alias("best_60min_w"),
        "rides", "ride_hours", "ride_km", "runs", "run_km", "workouts", "training_hours", "training_load",
        F.round("max_hr_bpm", 0).alias("max_hr_bpm"),
        F.when(F.col("best_5k_s").isNotNull(), duration_label(F.col("best_5k_s"))).alias("best_5k_time"),
        F.when(F.col("best_10k_s").isNotNull(), duration_label(F.col("best_10k_s"))).alias("best_10k_time"),
    )
)

# Fitness (CTL, 42-day), fatigue (ATL, 7-day) and form (TSB) from daily training load
daily_pd = (
    wk.groupBy("local_date").agg(F.sum(F.coalesce("training_load_score", F.lit(0.0))).alias("load")).toPandas()
)
rows = []
if len(daily_pd):
    loads = {r.local_date: float(r.load) for r in daily_pd.itertuples()}
    d, ctl, atl = min(loads), 0.0, 0.0
    while d <= today:
        load = loads.get(d, 0.0)
        tsb = ctl - atl
        ctl += (load - ctl) / 42.0
        atl += (load - atl) / 7.0
        rows.append((d, d.year, round(load, 1), round(ctl, 1), round(atl, 1), round(tsb, 1)))
        d += _dt.timedelta(days=1)
fitness_daily = spark.createDataFrame(rows, "date date, year int, training_load double, fitness_ctl double, fatigue_atl double, form_tsb double")

def form_status(tsb):
    if tsb is None: return "No data"
    if tsb > 5: return "Fresh"
    if tsb >= -10: return "Neutral"
    if tsb >= -30: return "Productive training"
    return "High fatigue"

last_day = rows[-1] if rows else (today, this_year, 0.0, None, None, None)
fy = {r["year"]: r for r in fitness_year.collect()}
recent = ride_power.where((F.col("duration_s") == 1200) & (F.col("local_date") >= F.date_sub(F.current_date(), 90))) \
    .agg(F.max("avg_power_w").alias("w"), F.max_by("local_date", "avg_power_w").alias("d")).collect()[0]
latest_ftp_year = max([y for y, r in fy.items() if r["ftp_w"]], default=None)
current_ftp = round(recent["w"] * FTP_FACTOR) if recent["w"] else (fy[latest_ftp_year]["ftp_w"] if latest_ftp_year else None)
ftp_this = fy.get(this_year, {}) and fy[this_year]["ftp_w"]
ftp_last = fy.get(last_year, {}) and fy[last_year]["ftp_w"]
summary = spark.createDataFrame([(
    float(current_ftp) if current_ftp else None,
    "95% of best 20-min power in last 90 days" if recent["w"] else f"95% of best 20-min power in {latest_ftp_year}",
    recent["d"],
    float(round(recent["w"])) if recent["w"] else None,
    float(ftp_this) if ftp_this else None,
    float(ftp_last) if ftp_last else None,
    float(round((ftp_this - ftp_last) / ftp_last, 3)) if ftp_this and ftp_last else None,
    str(this_year), str(last_year),
    last_day[3], last_day[4], last_day[5], form_status(last_day[5]),
    float(fy[this_year]["max_hr_bpm"]) if this_year in fy and fy[this_year]["max_hr_bpm"] else None,
)], "current_ftp_w double, current_ftp_method string, current_ftp_date date, best_20min_90d_w double, ftp_this_year_w double, "
   "ftp_last_year_w double, ftp_change_pct double, this_year string, last_year string, fitness_ctl double, fatigue_atl double, "
   "form_tsb double, form_status string, max_hr_this_year double")

min_year = this_year - FITNESS_YEARS_BACK
years = [y for y in range(min_year, this_year + 1)]
dim_year = spark.createDataFrame([(y, str(y)) for y in years], "year int, year_label string")
fitness_year = fitness_year.where(F.col("year") >= min_year)
power_curve = power_curve.where(F.col("year") >= min_year)
fitness_daily = fitness_daily.where(F.col("year") >= min_year)

save_gold(dim_year, GOLD_DIM_YEAR)
save_gold(fitness_year, GOLD_FITNESS_YEAR)
save_gold(power_curve, GOLD_POWER_CURVE)
save_gold(fitness_daily, GOLD_FITNESS_DAILY)
save_gold(summary, GOLD_FITNESS_SUMMARY)

for table_name in [GOLD_ACTIVITIES, GOLD_DAILY, GOLD_WEEKLY, GOLD_NUTRITION, GOLD_SAMPLES, GOLD_SEGMENTS, GOLD_ZONES, GOLD_BEST_EFFORTS]:
    spark.sql(f"OPTIMIZE delta.`Tables/{table_name}`")

for legacy in LEGACY_GOLD_TABLES:
    if spark.catalog.tableExists(legacy):
        spark.sql(f"DROP TABLE {legacy}")
        print(f"Dropped legacy Gold table {legacy}")

result = {
    "status": "success",
    "layer": "gold",
    "source_silver_rows": silver.count(),
    "gold_activities_rows": gold_activities.count(),
    "gold_daily_rows": daily_summary.count(),
    "gold_weekly_rows": weekly_metrics.count(),
    "gold_nutrition_rows": nutrition_recommendations.count(),
    "gold_sample_rows": samples.count(),
    "gold_segment_rows": segments.count(),
    "gold_zone_rows": zones.count(),
    "gold_best_effort_rows": best_efforts.count(),
    "gold_section_rows": sections.count(),
    "gold_insight_rows": insights.count(),
    "gold_power_curve_rows": power_curve.count(),
    "gold_fitness_year_rows": fitness_year.count(),
    "current_ftp_w": current_ftp,
}

print(json.dumps(result, indent=2))
weekly_metrics.orderBy(F.desc("week_start")).show(12, truncate=False)

notebookutils.notebook.exit(json.dumps(result))